In [18]:
import matplotlib.pyplot as plt 
import numpy as np 
import os 
import pandas as pd # 

In [19]:
df = pd.read_csv('match_data_50_tourns_modified.csv')
df.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage,tournament_id
0,Cao Yupeng,Jiang Jun,9,1411,1044,0.918016,0.714532,341,179,2302,...,242,32,17,32,17,5,0,0.0,1.000000,5808
1,Siripaporn Nuanthakhamjan,Zhou Yuelong,9,965,1384,0.056881,0.259705,3,0,23,...,4,245,118,808,430,2,5,1.0,0.285714,5808
2,Wu Yize,Allan Taylor,9,1342,1237,0.656987,0.565251,78,37,552,...,215,101,49,336,147,5,3,0.0,0.625000,5808
3,Ben Woollaston,Oliver Brown,9,1412,1146,0.845318,0.660383,803,454,5053,...,282,91,35,235,97,5,2,0.0,0.714286,5808
4,Andres Petrov,Mark Williams,9,1045,1611,0.017765,0.195447,51,18,304,...,61,281,170,1112,649,2,5,1.0,0.285714,5808


In [20]:
n = round(len(df) / 6)
df_holdout = df.tail(n)
df=df.iloc[:-n]

### Decision tree with statistical features

In [21]:
features_to_remove = ['player1', 'player2', 'player1_elo','player2_elo','elo_match_win_rate','elo_frame_win_rate',
                      'score1','score2','match_result','win_percentage','tournament_id']
X = df.drop(columns=features_to_remove) 
y = df['win_percentage']

In [22]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import make_scorer, root_mean_squared_error
import numpy as np

In [23]:
tscv = TimeSeriesSplit(n_splits=5)

model = DecisionTreeRegressor(random_state=20)
param_grid = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'ccp_alpha': [0.0, 0.01, 0.05]
}

rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=tscv,
    scoring=rmse_scorer
)

grid_search.fit(X, y)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=DecisionTreeRegressor(random_state=20),
             param_grid={'ccp_alpha': [0.0, 0.01, 0.05],
                         'max_depth': [3, 5, 10, 20, None],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'))

In [24]:
train_sizes = []
for train_index, _ in tscv.split(X):
    train_sizes.append(len(train_index))
train_sizes = np.array(train_sizes)

cv_results = grid_search.cv_results_
n_splits = tscv.get_n_splits()
n_params = len(cv_results['params'])

weighted_avg_rmse = []

for i in range(n_params):
    fold_scores = np.array([cv_results[f'split{j}_test_score'][i] for j in range(n_splits)])
    # weighted average by train sizes
    weighted_score = np.average(fold_scores, weights=train_sizes)
    weighted_avg_rmse.append(weighted_score)
best_index = np.argmax(weighted_avg_rmse)  # scores are negative RMSE, so max is best
best_params = cv_results['params'][best_index]
best_rmse = -weighted_avg_rmse[best_index]  # convert to positive RMSE

print("Best params (weighted):", best_params)
print("Best weighted average RMSE:", best_rmse)

Best params (weighted): {'ccp_alpha': 0.0, 'max_depth': 3, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best weighted average RMSE: 0.2868090931145643


### kNN with statistical features based on player statistic differences

In [25]:
dfm = df
dfm['p1_matches_win_ratio']=dfm['p1_matches_won']/df['p1_matches_played']
dfm['p2_matches_win_ratio']=dfm['p2_matches_won']/df['p2_matches_played']
dfm['p1_frames_win_ratio']=dfm['p1_frames_won']/df['p1_frames_played']
dfm['p2_frames_win_ratio']=dfm['p2_frames_won']/df['p2_frames_played']
dfm['p1_frames_win_ratio_1_year']=dfm['p1_frames_won_1_year']/df['p1_frames_played_1_year']
dfm['p2_frames_win_ratio_1_year']=dfm['p2_frames_won_1_year']/df['p2_frames_played_1_year']
dfm['p1_frames_win_ratio_3_years']=dfm['p1_frames_won_3_years']/df['p1_frames_played_3_years']
dfm['p2_frames_win_ratio_3_years']=dfm['p2_frames_won_3_years']/df['p2_frames_played_3_years']
dfm.fillna(0.5, inplace=True)
dfm['matches_win_ratio_diff']=dfm['p1_matches_win_ratio']-dfm['p2_matches_win_ratio']
dfm['frames_win_ratio_diff']=dfm['p1_frames_win_ratio']-dfm['p2_frames_win_ratio']
dfm['frames_win_ratio_diff_1_year']=dfm['p1_frames_win_ratio_1_year']-dfm['p2_frames_win_ratio_1_year']
dfm['frames_win_ratio_diff_3_years']=dfm['p1_frames_win_ratio_3_years']-dfm['p2_frames_win_ratio_3_years']

In [26]:
selected_features = ['matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']
X = dfm[selected_features]
y = dfm['win_percentage']

In [27]:
tscv = TimeSeriesSplit(n_splits=5)

model = DecisionTreeRegressor(random_state=20)
param_grid = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'ccp_alpha': [0.0, 0.01, 0.05]
}

rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=tscv,
    scoring=rmse_scorer
)

grid_search.fit(X, y)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=DecisionTreeRegressor(random_state=20),
             param_grid={'ccp_alpha': [0.0, 0.01, 0.05],
                         'max_depth': [3, 5, 10, 20, None],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'))

In [28]:
train_sizes = []
for train_index, _ in tscv.split(X):
    train_sizes.append(len(train_index))
train_sizes = np.array(train_sizes)
cv_results = grid_search.cv_results_
n_splits = tscv.get_n_splits()
n_params = len(cv_results['params'])

weighted_avg_rmse = []

for i in range(n_params):
    fold_scores = np.array([cv_results[f'split{j}_test_score'][i] for j in range(n_splits)])
    # weighted average by train sizes
    weighted_score = np.average(fold_scores, weights=train_sizes)
    weighted_avg_rmse.append(weighted_score)
best_index = np.argmax(weighted_avg_rmse)  # scores are negative RMSE, so max is best
best_params = cv_results['params'][best_index]
best_rmse = -weighted_avg_rmse[best_index]  # convert to positive RMSE

print("Best params (weighted):", best_params)
print("Best weighted average RMSE:", best_rmse)

Best params (weighted): {'ccp_alpha': 0.0, 'max_depth': 3, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best weighted average RMSE: 0.2824761889923819


### Decision tree with player elo ratings and features based on player statistic differences

In [29]:
selected_features = ['player1_elo','player2_elo','matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']
X = dfm[selected_features]
y = dfm['win_percentage']

In [30]:
tscv = TimeSeriesSplit(n_splits=5)

model = DecisionTreeRegressor(random_state=20)
param_grid = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'ccp_alpha': [0.0, 0.01, 0.05]
}

rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=tscv,
    scoring=rmse_scorer
)

grid_search.fit(X, y)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=DecisionTreeRegressor(random_state=20),
             param_grid={'ccp_alpha': [0.0, 0.01, 0.05],
                         'max_depth': [3, 5, 10, 20, None],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'))

In [31]:
train_sizes = []
for train_index, _ in tscv.split(X):
    train_sizes.append(len(train_index))
train_sizes = np.array(train_sizes)

cv_results = grid_search.cv_results_
n_splits = tscv.get_n_splits()
n_params = len(cv_results['params'])

weighted_avg_rmse = []

for i in range(n_params):
    fold_scores = np.array([cv_results[f'split{j}_test_score'][i] for j in range(n_splits)])
    # weighted average by train sizes
    weighted_score = np.average(fold_scores, weights=train_sizes)
    weighted_avg_rmse.append(weighted_score)
best_index = np.argmax(weighted_avg_rmse)  # scores are negative RMSE, so max is best
best_params = cv_results['params'][best_index]
best_rmse = -weighted_avg_rmse[best_index]  # convert to positive RMSE

print("Best params (weighted):", best_params)
print("Best weighted average RMSE:", best_rmse)

Best params (weighted): {'ccp_alpha': 0.0, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10}
Best weighted average RMSE: 0.2832591941549934


### Decision tree on all available features

In [32]:
features_to_remove = ['player1', 'player2', 
                      'score1','score2','match_result','win_percentage','tournament_id']
X = df.drop(columns=features_to_remove) 
y = df['win_percentage']

In [33]:
tscv = TimeSeriesSplit(n_splits=5)

model = DecisionTreeRegressor(random_state=20)
param_grid = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'ccp_alpha': [0.0, 0.01, 0.05]
}

rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=tscv,
    scoring=rmse_scorer
)

grid_search.fit(X, y)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=DecisionTreeRegressor(random_state=20),
             param_grid={'ccp_alpha': [0.0, 0.01, 0.05],
                         'max_depth': [3, 5, 10, 20, None],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'))

In [34]:
train_sizes = []
for train_index, _ in tscv.split(X):
    train_sizes.append(len(train_index))
train_sizes = np.array(train_sizes)

cv_results = grid_search.cv_results_
n_splits = tscv.get_n_splits()
n_params = len(cv_results['params'])

weighted_avg_rmse = []

for i in range(n_params):
    fold_scores = np.array([cv_results[f'split{j}_test_score'][i] for j in range(n_splits)])
    # weighted average by train sizes
    weighted_score = np.average(fold_scores, weights=train_sizes)
    weighted_avg_rmse.append(weighted_score)
best_index = np.argmax(weighted_avg_rmse)  # scores are negative RMSE, so max is best
best_params = cv_results['params'][best_index]
best_rmse = -weighted_avg_rmse[best_index]  # convert to positive RMSE

print("Best params (weighted):", best_params)
print("Best weighted average RMSE:", best_rmse)

Best params (weighted): {'ccp_alpha': 0.0, 'max_depth': 3, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best weighted average RMSE: 0.27571494794791557
